# 01 — Adoption ingest: Entra sign-in logs → Bronze

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 12 — T5 |
| **Layer** | `Bronze/adoption/YYYY-MM-DD/` |
| **Source** | Entra `SignInLogs` in `log-ihzhhpf-sit` (Log Analytics) |
| **Target** | `lh_ihzhhpf_sit` Bronze Delta table `bronze.adoption_signins` |
| **Spec** | `docs/superpowers/specs/2026-07-09-sprint-12-org-design.md` §7 |

Reads the last 24h of `ihzhhpf-app` sign-ins from Log Analytics and appends
them to Bronze. **No PHI** — sign-in metadata carries UPN + IP only; IP is
redacted to a /24. Scheduled nightly by `.github/workflows/adoption-refresh.yml`.

> When real telemetry has not yet accumulated, seed Bronze with
> `data-platform/scripts/adoption_seed_synthetic.py` (30-day backfill).


In [ ]:
LAKEHOUSE = "lh_ihzhhpf_sit"
BRONZE_TABLE = f"{LAKEHOUSE}.bronze.adoption_signins"
LOG_ANALYTICS_WORKSPACE_ID = "log-ihzhhpf-sit"  # resolved to workspace GUID at runtime
LOOKBACK = "24h"


## 1. Kusto query

Projects the Bronze adoption contract fields (design spec §7). `env` is derived
downstream in Silver from the app URL host; Bronze keeps the raw sign-in shape.


In [ ]:
KQL = f"""
SigninLogs
| where TimeGenerated > ago({LOOKBACK})
| where AppDisplayName startswith "ihzhhpf-app"
| project
    TimeGenerated,
    UserId,
    UserPrincipalName,
    AppDisplayName,
    AppId,
    ResultType,
    IPAddress,
    ClientAppUsed,
    DeviceDetail_TrustType = tostring(DeviceDetail.trustType),
    Location_CountryOrRegion = tostring(LocationDetails.countryOrRegion)
"""


## 2. Read from Log Analytics

Uses the Azure Monitor / Log Analytics Spark connector. Managed identity /
workload identity federation provides the token (no secrets in the notebook).


In [ ]:
df = (
    spark.read.format("com.microsoft.kusto.spark.synapse.datasource")
    .option("spark.synapse.linkedService", "log-ihzhhpf-sit")
    .option("query", KQL)
    .load()
)
df.printSchema()


## 3. Redact IP to /24 and land in Bronze

Idempotent per day: Bronze is append-only partitioned by ingest date; Silver
handles deduplication.


In [ ]:
from pyspark.sql import functions as F

bronze = (
    df
    .withColumn(
        "IPAddress",
        F.concat_ws(".", F.split("IPAddress", "\\.").getItem(0),
                    F.split("IPAddress", "\\.").getItem(1),
                    F.split("IPAddress", "\\.").getItem(2), F.lit("0"))
    )
    .withColumn("ingestDate", F.current_date())
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LAKEHOUSE}.bronze")
(
    bronze.write.format("delta")
    .mode("append")
    .partitionBy("ingestDate")
    .saveAsTable(BRONZE_TABLE)
)
print(f"Appended {bronze.count()} sign-in rows to {BRONZE_TABLE}")
